# AI Contract Lens — training and evaluation

This notebook trains three models on the supplied OPP-115 policy clauses: TF-IDF + Logistic Regression, a BiLSTM without attention, and a from-scratch BiLSTM with trainable embeddings and attention. It uses only the seven labels derived from the human-annotated OPP-115 categories and splits by `website_id` so a website cannot occur in more than one partition.

**No synthetic data or labels are used.** The cookie-banner CSV is excluded because its labels are explicitly marked heuristic and review-required. Metrics below are produced only when the training/evaluation cells finish; do not report unexecuted, sample, or expected values.

In Colab/Kaggle, upload or clone the project so this notebook runs from the project root and ensure the prepared CSV is accessible. Select a GPU runtime if available; training automatically uses CUDA when PyTorch detects it. The OPP-115 corpus has research/teaching and non-commercial terms; review the supplied corpus documentation before redistributing or using it commercially.

In [ ]:
# Run from the project root. For Colab/Kaggle, change directory to the uploaded/cloned folder first.
%pip install -e .


In [ ]:
import os
from pathlib import Path

import torch

PROJECT_DIR = Path.cwd()
DATA_PATH = Path(os.environ.get(
    "CONTRACT_LENS_DATA",
    PROJECT_DIR / "Working Prototype" / "opp115_policy_clauses_prepared.csv",
))
ARTIFACT_DIR = Path(os.environ.get("CONTRACT_LENS_ARTIFACTS", PROJECT_DIR / "artifacts"))
SEED = 42
EPOCHS = 12
BATCH_SIZE = 64

print("Project directory:", PROJECT_DIR)
print("Dataset:", DATA_PATH)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. Set CONTRACT_LENS_DATA to the prepared CSV path."
    )


## Train and evaluate

The reusable training code cleans the supplied text, validates every prepared label against its original OPP-115 category, removes exact duplicate clause/category rows, builds a vocabulary from training data only, applies class weights to the neural losses, and selects neural checkpoints using validation macro-F1. The test partition remains held out by website.

In [ ]:
from contract_lens.training import train

metrics = train(
    dataset_path=DATA_PATH,
    artifact_dir=ARTIFACT_DIR,
    seed=SEED,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)
print("Device:", metrics["device"])
print("Split rows:", metrics["split_sizes"])
print("Split website counts:", metrics["split_websites"])
print("Artifacts:", ARTIFACT_DIR.resolve())


## Measured test results and diagnostics

The table and class reports below reflect the just-completed held-out test evaluation. Macro-F1 weights labels equally and is important because class frequencies are imbalanced. Confusion matrices use true classes as rows and predicted classes as columns. Error rows include source clauses for manual review.

In [ ]:
import pandas as pd

model_rows = []
for model_name, scores in metrics["models"].items():
    model_rows.append({
        "Model": model_name,
        "Accuracy": scores["accuracy"],
        "Macro precision": scores["macro_precision"],
        "Macro recall": scores["macro_recall"],
        "Macro-F1": scores["macro_f1"],
        "Weighted-F1": scores["weighted_f1"],
    })
display(pd.DataFrame(model_rows).set_index("Model").round(4))

for model_name, scores in metrics["models"].items():
    print("\nPer-class test report —", model_name)
    display(pd.DataFrame(scores["per_class"]).T.round(4))
    matrix_path = ARTIFACT_DIR / f"confusion_matrix_{model_name}.csv"
    if matrix_path.is_file():
        print("Confusion matrix — rows=true, columns=predicted")
        display(pd.read_csv(matrix_path, index_col=0))
    errors_path = ARTIFACT_DIR / f"errors_{model_name}.csv"
    if errors_path.is_file():
        errors = pd.read_csv(errors_path)
        print(f"Held-out error analysis ({len(errors)} rows) —", model_name)
        display(errors.head(20))

print("Only these measured values from this completed run should be reported.")
